# 04 · The deployed app: Lakebase dashboard, invoice check, preparation plan (executed)

Calls the **deployed Databricks App** exactly as the browser does (OAuth). Shows:
1. the Live Dashboard served from **Lakebase** (latency over 5 calls), including the morning brief and ML forecast;
2. the invoice check on the demo PDF through **Unity AI Gateway** (parse only, no approval);
3. the preparation plan: gold SQL + Lakebase + ML forecast + MCP weather tool via the gateway.

In [1]:
import warnings; warnings.filterwarnings('ignore')
from proof_helpers import *
print('catalog.schema:', FQ)

catalog.schema: fevm_cme_conde_catalog.kaapi_bricks


In [2]:
st = hq.apps.get(APP_NAME)
print("app:", st.name, "| app status:", st.app_status.state.value, "| compute:", st.compute_status.state.value)
r = app_call("GET", "/api/stores"); stores = r.json().get("stores", r.json()) if r.ok else []
print("GET /api/stores ->", r.status_code, "|", len(stores), "stores, e.g.", list(stores)[:3])

app: kaapi-bricks-finale | app status: RUNNING | compute: ACTIVE


GET /api/stores -> 200 | 37 stores, e.g. ['Koramangala, Bangalore', 'Indiranagar, Bangalore', 'Jayanagar, Bangalore']


## 1 · Live Dashboard from Lakebase

In [3]:
import statistics
lat, server = [], []
for _ in range(5):
    t = time.time(); d = app_call("GET", "/api/lakebase/dashboard", params={"store": "Koramangala, Bangalore"}).json()
    lat.append((time.time() - t) * 1000); server.append(d["query_ms"])
print(f"Lakebase query time (6 queries + brief + forecast), ms: min {min(server)}  median {statistics.median(server)}  max {max(server)}")
print(f"round trip from this notebook, ms: median {statistics.median(lat):.0f}")
print("KPIs:", json.dumps(d["kpis"], default=str)[:400])
print("\nToday's action plan:", d["brief"]["headline"], f"(numbers verified {d['brief']['numbers_grounded']}/{d['brief']['numbers_checked']})")
for a in d["brief"]["actions"]:
    print(f"  {a.get('priority')}. {a['action']}  — {a['why']}")
print("\nToday's demand forecast (model vs usual for this weekday):")
print(pd.DataFrame(d["forecast_today"])[["product_name", "forecast_units", "baseline_same_weekday"]].to_string(index=False))

Lakebase query time (6 queries + brief + forecast), ms: min 29.9  median 33.1  max 38.1
round trip from this notebook, ms: median 1674
KPIs: {"below_reorder": 0, "lowest_cover": {"ingredient_id": "ING-002", "ingredient_name": "Chikmagalur Robusta Beans", "unit": "kg", "current_stock": 69.93, "reorder_threshold": 15.0, "below_reorder": false, "days_cover": 18.2, "avg_daily_usage": 3.836, "last_txn_date": "2026-07-31"}, "overdue_pos": 8, "overdue_value": 31065.06, "late_deliveries": 9, "avg_days_late": 1.6, "units_7d": 744}

Today's action plan: Friday push: chase 8 overdue POs worth ₹31065 and prep for above-average demand. (numbers verified 23/23)
  1. Escalate PO-00816 with Mysore Sweet Works; payment of ₹6017 is 182 days overdue.  — PO-00816 to Mysore Sweet Works is the oldest overdue at 182 days, risking supplier relationship.
  2. Follow up PO-00951 with Mysore Sweet Works for ₹4967 overdue 136 days.  — Second overdue PO to same supplier; 8 total overdue POs worth ₹31065 need urgen

## 2 · Invoice check through Unity AI Gateway (demo invoice for PO-01057, parse only)

In [4]:
t = time.time()
with open("../sample-invoices/invoice_mysore_po01057.pdf", "rb") as f:
    r = app_call("POST", "/api/parse-invoice", files={"file": ("invoice_mysore_po01057.pdf", f, "application/pdf")},
                 data={"store": "Koramangala, Bangalore"})
res = r.json()
print("HTTP", r.status_code, f"| {time.time()-t:.1f} s round trip | server {res.get('latency_ms', 0)/1000:.1f} s")
ex = res["extracted"]
print("extracted by kaapi_llm (gateway):", {k: ex.get(k) for k in ("supplier_name", "invoice_number", "invoice_date", "po_reference", "total_amount")})
print("matched PO:", (res.get("po_match") or {}).get("po_id"), "| warnings:", res.get("warnings"))
items = pd.DataFrame(res["matched_items"])
cols = [c for c in ("invoice_name", "invoice_qty", "po_qty", "invoice_rate", "po_rate", "status", "issues") if c in items]
print(items[cols].to_string(index=False))
print("\nstatus counts:", items.status.value_counts().to_dict())

HTTP 200 | 8.4 s round trip | server 7.1 s
extracted by kaapi_llm (gateway): {'supplier_name': 'Mysore Sweet Works', 'invoice_number': 'MSW-INV-2026-0729', 'invoice_date': '2026-07-29', 'po_reference': 'PO-01057', 'total_amount': 4761.44}
matched PO: PO-01057 | warnings: []
  invoice_name  invoice_qty  po_qty  invoice_rate  po_rate  status                                           issues
   Badam Paste         2.15    2.15         800.0    800.0   match                                               []
    Rose Syrup         6.39    6.39         230.0    200.0   error [Price: contract ₹200.0, invoiced ₹230.0 (+₹30)]
Jaggery Blocks        14.00   12.69          80.0     80.0 warning       [Qty: ordered 12.69, invoiced 14.0 (+1.3)]
         Sugar         5.00     NaN          45.0      NaN   extra                         [Not in PO — extra item]
  Palm Jaggery          NaN    5.76           NaN    150.0 missing    [In PO but not in invoice — missing delivery]

status counts: {'match': 1, 

## 3 · Preparation plan (gold SQL + Lakebase + ML forecast + MCP weather via the gateway)

In [5]:
r = app_chat("How should I prepare for today?")
print(f"{r['seconds']} s | trace {r.get('trace_id')} | cached={r.get('from_cache')}")
print(mask(r.get("full_text", ""))[:3000])

27.2 s | trace tr-2fd2dc1ece7a33e25b5aab90793d6229 | cached=False
# ☕ Kaapi Bricks Koramangala (STR-001) — Preparation Plan
**Saturday, 26 October 2026** | Generated by Kaapi Bricks Operations Advisor

---

## 📋 Situation Summary

| Factor | Detail |
|---|---|
| **Day Type** | Saturday (Weekend) |
| **Holiday** | None |
| **Weather** | Moderate drizzle, 20.6–27.3°C |
| **Forecast Basis** | ML model (UC demand_forecaster v2) + weather adjustment |

> **How forecasts are built:** The ML model already accounts for weekday/weekend patterns, holidays, and recent demand trends. The same-weekday averages are shown for reference. The only manual adjustment applied here is for **weather**, which the model does not include.

---

## 🌦️ Weather Impact Assessment

**Condition: Moderate Drizzle | Low: 20.6°C | High: 27.3°C**

Bangalore drizzle on a Saturday is a **net positive** for footfall — customers seek warm shelter and a hot cup. Expect:

| Signal | Impact |
|---|---|
| 🌧️ Rain/drizzle | **Ho